In [ ]:
! pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00


In [ ]:
data = [
 {
        "instruction":"What is AI?",
        "response":"Artificial Intelligence is the simulation of human intelligence by machines."
    },{
        "instruction":"What is Machine Learning?",
        "response":"Machine Learning is a subset of Artificial Intelligence."
    },{
        "instruction":"What is Deep Learning?",
        "response":"Deep Learning uses neural networks with many hidden layers."
    },{
        "instruction":"Who developed Python?",
        "response":"Python was developed by Guido van Rossum."
    },
 {
        "instruction":"What is NLP?",
        "response":"Natural Language Processing enables computers to understand human language."
    }]
print(data)

[{'instruction': 'What is AI?', 'response': 'Artificial Intelligence is the simulation of human intelligence by machines.'}, {'instruction': 'What is Machine Learning?', 'response': 'Machine Learning is a subset of Artificial Intelligence.'}, {'instruction': 'What is Deep Learning?', 'response': 'Deep Learning uses neural networks with many hidden layers.'}, {'instruction': 'Who developed Python?', 'response': 'Python was developed by Guido van Rossum.'}, {'instruction': 'What is NLP?', 'response': 'Natural Language Processing enables computers to understand human language.'}]


In [ ]:
# 3 CONVERT INTO HUGGIN FACE DATASET - BCZ PYTHON LIST IS NOT COMPATIBLE
from datasets import Dataset

dataset = Dataset.from_list(data)
print(dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 5
})


In [ ]:
print(dataset[0])

{'instruction': 'What is AI?', 'response': 'Artificial Intelligence is the simulation of human intelligence by machines.'}


In [ ]:
# 5 fomrat columns : convert the entire thing to text - Datasets format to text


def format_data(example):
  text = f'''
   ## instruction : {example['instruction']}

   ## response : {example['response']}
  '''
  example['text'] = text
  return example

dataset = dataset.map(format_data)
print(dataset['text'][0])


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

 
   ## instruction : What is AI?

   ## response : Artificial Intelligence is the simulation of human intelligence by machines.
  


In [ ]:
# 7 load tokenizer

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')



config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
# 8 set PAD tokens, gpt 2 doenst have padding , we use EOS instead of <Pad>


tokenizer.pad_token = tokenizer.eos_token




In [ ]:
def tokenize(example):
  return tokenizer(
      example['text'],
      truncation = True,
      padding = 'max_length',
      max_length = 128

  )

tokenized_dataset = dataset.map(tokenize)


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset[0]['input_ids'])

[220, 198, 220, 220, 22492, 12064, 1058, 1867, 318, 9552, 30, 628, 220, 220, 22492, 2882, 1058, 35941, 9345, 318, 262, 18640, 286, 1692, 4430, 416, 8217, 13, 198, 220, 220, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]


In [ ]:
from transformers import AutoModelForCausalLM


model =AutoModelForCausalLM.from_pretrained('gpt2')

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# create labels for causal language modeling
# In causal language modeling, the model predicts the next token in a sequence.
# Therefore, the input sequence itself serves as the target labels for training.
def add_labels(example):
  example['labels'] = example['input_ids'].copy()

  return example

tokenized_dataset = tokenized_dataset.map(add_labels)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset)

Dataset({
    features: ['instruction', 'response', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 5
})


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = './model',
    num_train_epochs = 20 ,
    per_device_train_batch_size = 2,
    learning_rate = 5e-5,
    logging_steps = 1,
    save_strategy = 'epoch',
    report_to = 'none'

)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_dataset,

)

In [ ]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,8.029940
2,3.579022
3,1.782256
4,1.169411
5,1.301887
6,1.163925
7,1.209816
8,1.038936
9,1.035000
10,0.861068


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=30, training_loss=1.0434412638346353, metrics={'train_runtime': 139.7103, 'train_samples_per_second': 0.358, 'train_steps_per_second': 0.215, 'total_flos': 3266150400000.0, 'train_loss': 1.0434412638346353, 'epoch': 10.0})

In [ ]:
trainer.save_model('my_model')
tokenizer.save_pretrained('my_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('my_model/tokenizer_config.json', 'my_model/tokenizer.json')

In [ ]:
# 17  test the model

from transformers import pipeline

generator = pipeline('text-generation',
                     model = 'my_model',
                     tokenizer = 'my_model')

prompt = ''' ## instruction : who developed python ?  '''

output = generator(
    prompt,
    max_new_tokens = 30
)

print(output[0]['generated_text'])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 ## instruction : who developed python ?    ## response : python is a language developed by humans.  


In [ ]:
# here we will descuss about once the LLMS are fine tuned then we need to check
# how well they are trained using some evaluation metrics
# common evaluation metrics of LLMS are
BLEU
ROUGE
BERTscore
perplexity




In [ ]:
!pip install -q evaluate torchao --upgrade
import numpy as np
import evaluate
from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.3 MB/s eta 0:00:00


In [ ]:
dataset = load_dataset('stanfordnlp/imdb')

model_name = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(example['text'],
                     truncation=True,
                     padding='max_length',
                     max_length = 256)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Role of Alpha in LoRA

The `alpha` parameter acts as a multiplier for the combined low-rank matrices before they are added to the original weights. Mathematically, if $W_0$ is the original weight matrix, $A$ and $B$ are LoRA matrices of rank $r$, the adapted weights become:

$$ W_{\text{adapted}} = W_0 + \frac{\alpha}{r} (A \cdot B) $$

*   **Scaling effect**: Alpha controls the magnitude of the updates contributed by the low-rank matrices. A higher alpha leads to more pronounced changes in the output, while a lower alpha results in subtler modifications.

*   **Stability and convergence**: Adjusting alpha affects the learning dynamics—too high an alpha can cause instability or overshooting during fine-tuning, while too low might result in slow or insufficient model adaptation.

*   **Interaction with rank**: Alpha commonly scales inversely with the chosen rank, as shown by the division by $r$ in implementations, ensuring that increased rank doesn’t disproportionately amplify weight changes.

### Practical Considerations

*   **Choosing alpha**: Many frameworks set alpha proportionally to rank, for example, alpha = 16 for rank = 8. The optimal value depends on task complexity, model size, and dataset characteristics.

*   **Impact on output**: Alpha indirectly balances how much the model leans on the learned low-rank adaptation versus the frozen pretrained weights. Fine-tuning tasks requiring stronger domain adaptation might benefit from a higher alpha.

*   **Efficiency**: Using alpha with low-rank matrices preserves the memory efficiency advantage of LoRA while enabling flexible control over adaptation strength.

In [ ]:
# Load a pre-trained model for sequence classification.
# `AutoModelForSequenceClassification` is a generic class that automatically selects
# the correct model architecture based on the `model_name` (e.g., 'distilbert-base-uncased').
# `num_labels=2` specifies that our task is a binary classification (e.g., positive/negative sentiment).
# The model will output two logits, one for each class.
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Initialize a DataCollator for batching and dynamic padding.
# `DataCollatorWithPadding` is crucial for efficiently processing batches of sequences.
# It takes care of padding sequences to the maximum length within each batch,
# ensuring that all sequences in a batch have the same length without padding to the global maximum,
# which saves computation and memory. It uses the provided `tokenizer` to determine padding tokens.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
)

# Configure LoRA (Low-Rank Adaptation) for efficient fine-tuning.
# LoRA is a parameter-efficient fine-tuning (PEFT) method that reduces the number of trainable parameters
# during fine-tuning, making the process faster and less memory-intensive.
lora_config = LoraConfig(
    task_type = TaskType.SEQ_CLS, # Specify the task type as Sequence Classification.
    r = 8 ,                      # 'r' is the rank of the update matrices. A smaller 'r' means fewer parameters to train.
    lora_alpha = 16,             # 'lora_alpha' is a scaling factor for the LoRA updates.
    lora_dropout = 0.1,          # 'lora_dropout' applies dropout to the LoRA layers to prevent overfitting.
    # `target_modules` specifies which layers of the pre-trained model will be modified with LoRA layers.
    # 'q_lin' and 'v_lin' typically refer to the query and value linear layers in the attention mechanism
    # of transformer models like DistilBERT, which are common targets for LoRA.
    target_modules = ['q_lin','v_lin']
)


# Apply the LoRA configuration to the base model.
# `get_peft_model` wraps the original model with LoRA layers, effectively freezing the original model weights
# and only training the newly added LoRA parameters.
model = get_peft_model(model , lora_config)
# Print a summary of the trainable parameters.
# This will show a significantly smaller number of trainable parameters compared to the total parameters
# of the full pre-trained model, demonstrating the efficiency of LoRA.
model.print_trainable_parameters()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [ ]:
# Load the 'accuracy' metric from the `evaluate` library.
# This object will be used later in `compute_metrics` to calculate the accuracy of our model's predictions.
accuracy = evaluate.load('accuracy')

# Define the `compute_metrics` function, which is called during evaluation to calculate performance metrics.
# This function receives `eval_pred`, which is a tuple containing the model's raw output (logits) and the true labels.
def compute_metrics(eval_pred):
  # Unpack the `eval_pred` tuple into logits (model's raw scores) and true labels.
  logits , labels = eval_pred
  # Convert logits into class predictions.
  # `np.argmax(logits, axis=-1)` takes the raw scores for each class (logits) and returns the index
  # of the highest score along the last dimension (`axis=-1`). For binary classification, this effectively
  # selects '0' or '1' as the predicted class based on which logit is higher.
  preds = np.argmax(logits,axis=-1)
  # Calculate and return the accuracy using the loaded 'accuracy' metric.
  # It compares the model's predictions (`preds`) with the true labels (`references`).
  return accuracy.compute(
      predictions = preds ,
      references = labels)

# Configure the training arguments for the Hugging Face Trainer.
# `TrainingArguments` defines all the hyperparameters for training and evaluation.
training_args = TrainingArguments(
    output_dir='./lora_output',          # Directory where model checkpoints and logs will be saved.
    num_train_epochs = 20,              # Total number of training epochs to perform (updated to 20).
    per_device_train_batch_size = 8,    # Batch size per GPU/TPU core for training.
    per_device_eval_batch_size = 8,     # Batch size per GPU/TPU core for evaluation.
    learning_rate = 2e-4,               # The initial learning rate for the AdamW optimizer.
    weight_decay = 0.01,                # Strength of weight decay (L2 regularization).
    logging_steps = 100,                # Number of updates steps between two logs.
    eval_strategy = 'epoch',            # Evaluation is performed once at the end of each epoch.
    save_strategy = 'epoch',            # Model checkpoints are saved at the end of each epoch.
    fp16 = True,                        # Enable mixed-precision training (using float16) for faster training and less memory usage.
    report_to = 'none'                  # Do not report training metrics to any online service.
)

# Initialize the Hugging Face Trainer.
# The Trainer orchestrates the entire training and evaluation loop.
trainer = Trainer(
    model = model,                      # The pre-trained model (with LoRA adaptation) to be trained.
    args = training_args,               # The training arguments defined above.
    train_dataset = tokenized_dataset['train'], # The dataset to use for training.
    eval_dataset = tokenized_dataset['test'],   # The dataset to use for evaluation.
    data_collator = data_collator,      # The data collator responsible for batching and padding.
    compute_metrics = compute_metrics   # The function to use for calculating metrics during evaluation.
)

In [ ]:
trainer.train()

results = trainer.evaluate()

print(results)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.267172,0.293296,0.893800
2,0.276502,0.254748,0.900680
3,0.239560,0.260042,0.907880
4,0.234293,0.267765,0.908480
5,0.172520,0.315222,0.909600
6,0.163166,0.298474,0.909280
7,0.142622,0.358144,0.909240
8,0.184898,0.347901,0.907720
9,0.141983,0.422158,0.906760
10,0.173492,0.433533,0.908200


Epoch,Training Loss,Validation Loss,Accuracy
1,0.267172,0.293296,0.893800
2,0.276502,0.254748,0.900680
3,0.239560,0.260042,0.907880
4,0.234293,0.267765,0.908480
5,0.172520,0.315222,0.909600
6,0.163166,0.298474,0.909280
7,0.142622,0.358144,0.909240
8,0.184898,0.347901,0.907720
9,0.141983,0.422158,0.906760
10,0.173492,0.433533,0.908200


Training Loss,Validation Loss,Epoch,Accuracy
0.073459,0.710288,20,0.907360


{'eval_loss': 0.7102879881858826, 'eval_accuracy': 0.90736}


In [ ]:
model.save_pretrained(
    'lora_sentiment_adapter'
)


NameError: name 'model' is not defined

In [ ]:
import torch

text = 'the movie is fast and energetic, more story ,engaging , good  making story to people'

inputs = tokenizer(text , return_tensors = 'pt',
                   truncation = True ,
                   padding = True)
inputs = {k:v.to(model.device)
            for k,v in inputs.items()
            }

model.eval()
with torch.no_grad():
  output = model(**inputs)

predictions = output.logits.argmax(dim=1).item()
print(predictions)





0


In [ ]:
import torch # Import the PyTorch library, essential for tensor operations and GPU acceleration.

# Define a list of movie reviews to test the trained model.
# Each string in the list is a review for which we want to predict the sentiment.
review = [
    'this movie was fantastic ...',
    ' the movie is fast and energetic, more story ,engaging , good  making story to people'
]

# Loop through each review in the 'review' list.
for text_review in review:
    # Tokenize the current review using the pre-trained tokenizer.
    # 'return_tensors="pt"' ensures the output is PyTorch tensors.
    # 'truncation=True' cuts off longer sequences to the maximum length the model can handle.
    # 'padding=True' adds padding to shorter sequences to match the maximum length.
    inputs = tokenizer(text_review, return_tensors='pt',
                       truncation=True,
                       padding=True)

    # Move the input tensors (input_ids, attention_mask) to the same device as the model (e.g., GPU if available).
    # This is crucial for consistent computations, especially during inference.
    inputs = {k: v.to(model.device)
              for k, v in inputs.items()}

    # Set the model to evaluation mode.
    # This disables dropout and batch normalization updates, which are only needed during training.
    model.eval()

    # Disable gradient calculation for inference.
    # This reduces memory consumption and speeds up computation, as no backpropagation is needed.
    with torch.no_grad():
        # Pass the tokenized input through the model to get predictions (logits).
        output = model(**inputs)

    # Extract the predicted class by finding the index with the highest logit value.
    # 'argmax(dim=1)' finds the index of the maximum value along the dimension of classes.
    # '.item()' converts the single-element tensor to a standard Python integer.
    prediction = output.logits.argmax(dim=1).item()

    # Map the numerical prediction (0 or 1) to a human-readable sentiment label.
    # Assuming 1 corresponds to 'Positive' and 0 to 'Negative'.
    sentiment = "Positive" if prediction == 1 else "Negative"

    # Print the original review and its predicted sentiment.
    print(f"Review: '{text_review}' -> Sentiment: {sentiment}")

Review: 'this movie was fantastic ...' -> Sentiment: Negative
Review: ' the movie is fast and energetic, more story ,engaging , good  making story to people' -> Sentiment: Negative


### Step-by-Step Inference for a Single Sentence

Let's pick a single sentence and see how the model processes it to predict sentiment. We'll examine the intermediate steps in detail.

In [ ]:
# 1. Define a test sentence
test_sentence = "This movie was an absolute joy to watch!"
print(f"Test Sentence: '{test_sentence}'")

Test Sentence: 'This movie was an absolute joy to watch!'


### 2. Tokenize the sentence

First, we tokenize the sentence. This converts the text into numerical IDs that the model can understand. We'll also specify `return_tensors='pt'` to get PyTorch tensors, `truncation=True` to handle long sentences, and `padding=True` to make all inputs the same length.

In [ ]:
inputs = tokenizer(test_sentence, return_tensors='pt', truncation=True, padding=True)
print("Tokenized Inputs (on CPU by default):\n", inputs)

Tokenized Inputs (on CPU by default):
 {'input_ids': tensor([[ 101, 2023, 3185, 2001, 2019, 7619, 6569, 2000, 3422,  999,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


### 3. Move input tensors to the model's device

Models typically run on a specific device (e.g., GPU for faster computation). The input tensors must be on the *same device* as the model for calculations to occur. This step moves the tokenized inputs to `model.device`.

In [ ]:
inputs = {k: v.to(model.device) for k, v in inputs.items()}
print(f"Model is on device: {model.device}")
print("Inputs after moving to model's device:\n", inputs)

Model is on device: cuda:0
Inputs after moving to model's device:
 {'input_ids': tensor([[ 101, 2023, 3185, 2001, 2019, 7619, 6569, 2000, 3422,  999,  102]],
       device='cuda:0'), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


### 4. Set the model to evaluation mode

Before making predictions, we set the model to `eval()` mode. This is important because it deactivates layers like dropout and batch normalization, which behave differently during training versus inference to ensure consistent predictions.

In [ ]:
model.eval()
print("Model set to evaluation mode.")

Model set to evaluation mode.


### 5. Perform inference without gradient calculation

We wrap the prediction call in `torch.no_grad()`. This tells PyTorch not to compute or store gradients during this operation, which saves memory and speeds up the process, as we don't need to update model weights during inference.

In [ ]:
with torch.no_grad():
    output = model(**inputs)
print("Model output (logits):\n", output.logits)

Model output (logits):
 tensor([[-3.3301,  3.6758]], device='cuda:0')


### 6. Extract the prediction

The `output.logits` tensor contains the raw prediction scores for each class (e.g., positive/negative). We use `argmax(dim=1)` to find the index of the class with the highest score, which represents the model's predicted class. `.item()` converts the tensor to a Python integer.

In [ ]:
prediction = output.logits.argmax(dim=1).item()
print(f"Raw prediction (class index): {prediction}")

Raw prediction (class index): 1


### 7. Map the prediction to a sentiment label

Finally, we convert the numerical class index into a human-readable sentiment label (e.g., 0 for Negative, 1 for Positive, assuming this mapping from the dataset).

In [ ]:
sentiment = "Positive" if prediction == 1 else "Negative"
print(f"Predicted Sentiment: {sentiment}")

Predicted Sentiment: Positive


In [ ]:
from transformers import pipeline
# Example: Sentiment analysis with BERT
classifier = pipeline("sentiment-analysis")
result = classifier("the movie is fast and energetic, more story ,engaging , good  making story to people")
print(result) # Output: [{'label': 'POSITIVE', 'score': 0.99}]

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998840093612671}]


# Q lora - finetuning

In [ ]:
## QLoRA: Quantized Low-Rank Adaptation Explained

QLoRA (Quantized Low-Rank Adaptation) is an efficient fine-tuning method that significantly reduces memory usage while maintaining performance comparable to full fine-tuning. It builds upon LoRA by introducing **quantization** of the pre-trained base model weights.

### The "Funda" of QLoRA: How Quantization Works

Your understanding is partially correct, and here's the clarification:

1.  **Base Model Quantization**: In QLoRA, the large pre-trained language model (LLM) is indeed **quantized to a lower precision**, typically **4-bit NormalFloat (NF4)**. This is a key distinguishing factor from standard LoRA. These 4-bit quantized base model weights are then **frozen** and *not* directly trained.

2.  **LoRA Adapter Addition**: After the base model is quantized and frozen, **LoRA (Low-Rank Adaptation) adapters are added** to its key linear layers (e.g., query, key, value, output projections in transformers).

3.  **LoRA Adapter Precision**: The LoRA adapter matrices (A and B) are *not* quantized to 4-bit. They are generally kept in **higher precision**, such as `fp16` (half-precision floating-point) or `bf16` (bfloat16). These LoRA adapters are the *only* parameters that are trained during fine-tuning.

4.  **Computational Flow during Training/Inference**:
    *   During the forward pass, the 4-bit quantized base model weights are **de-quantized on-the-fly** to a higher precision (e.g., `bf16`) for computation. This de-quantization happens dynamically as needed, not storing the full precision weights in memory.
    *   The results from the de-quantized base model weights are then combined with the output of the LoRA adapters (which are trained in `fp16`/`bf16`).
    *   The backward pass (gradient calculation for training) only computes gradients for the small, high-precision LoRA adapters.

So, to directly answer your question:

*   **Is it like loading in quantized form and then adding LoRA?** Yes, the base model is loaded in a quantized (e.g., 4-bit) and frozen state.
*   **Are only the low-rank matrices quantized?** No, the *base model* is quantized. The *low-rank matrices (LoRA adapters)* are typically kept in a higher precision and are the trainable part.


This approach allows QLoRA to achieve memory savings because the vast majority of the model's parameters (the base model weights) are stored in 4-bit, while still enabling efficient and effective fine-tuning by training only a small set of high-precision LoRA parameters.

### Detailed Flow of QLoRA

Here's a step-by-step breakdown of the QLoRA workflow:

1.  **Load Base Model (4-bit Quantization)**:
    *   The large pre-trained model (e.g., Llama 2, Falcon) is loaded with 4-bit quantization, typically using `bitsandbytes` library functionalities. This means the model weights are stored in 4-bit NormalFloat (NF4) format in memory. This greatly reduces the memory footprint.
    *   The base model's weights are then **frozen**; they will not be updated during training.

2.  **Add LoRA Adapters**:
    *   A `LoraConfig` is defined, specifying parameters like `r` (rank), `lora_alpha`, `lora_dropout`, and `target_modules` (which layers to apply LoRA to, typically attention and feed-forward layers' linear projections).
    *   These LoRA adapters (small, low-rank matrices) are attached to the frozen, 4-bit quantized base model layers. These adapters are initialized in a higher precision (e.g., `fp16` or `bf16`).

3.  **Setup for Training**:
    *   The model is wrapped using PEFT (`get_peft_model`) to ensure only the LoRA adapters are trainable.
    *   A training loop is set up, often using the Hugging Face `Trainer`.
    *   The data is tokenized and prepared for training.

4.  **Forward Pass (Training and Inference)**:
    *   When an input passes through a layer with LoRA adapters:
        *   The 4-bit quantized base weights for that layer are **de-quantized on-the-fly** to a higher precision (`bf16` is commonly used for computation).
        *   The input is multiplied by the de-quantized base weights.
        *   Simultaneously, the input is passed through the LoRA adapters (which are in `fp16`/`bf16`). The outputs of the LoRA adapters are scaled by `lora_alpha / r`.
        *   The output from the de-quantized base weights and the scaled LoRA adapter output are summed together.

5.  **Backward Pass (Training Only)**:
    *   During backpropagation, gradients are only calculated and applied to the **LoRA adapter weights**. The base model weights remain frozen and untouched.
    *   This significantly reduces the computational cost and memory required for gradient storage.

6.  **Save and Merge Adapters**:
    *   After fine-tuning, only the small LoRA adapter weights are saved.
    *   For deployment or further use, these LoRA adapters can be **merged back into the full-precision base model** weights (effectively reversing the process by adding `LoRA_A @ LoRA_B` to the original full-precision weights). This creates a new full-precision fine-tuned model that can be used without the PEFT library.